# Tutorial 04a: Logistic Regression

Mathematics for Machine Learning

Thomas Manke  
2026-11-19

## Setup

Run the setup below to load the <a
href="https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic"
target="_blank" rel="noopener">Breast Cancer Dataset</a> from the UCI
Machine Learning Repository using `fetch_ucirepo`.

In [1]:
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    log_loss,
    accuracy_score
)

sns.set_theme(context="notebook", style="whitegrid")
FIGSIZE = (7, 4)
RANDOM_STATE = 42

# Load Wisconsin Breast Cancer Dataset via fetch_ucirepo (UCI ID: 17)
try:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=17)
    # Standardize feature names to match scikit-learn standard descriptive names
    feature_mapping = {
        'radius1': 'mean radius', 'texture1': 'mean texture', 'perimeter1': 'mean perimeter', 'area1': 'mean area',
        'smoothness1': 'mean smoothness', 'compactness1': 'mean compactness', 'concavity1': 'mean concavity',
        'concave_points1': 'mean concave points', 'symmetry1': 'mean symmetry', 'fractal_dimension1': 'mean fractal dimension',
        'radius2': 'radius error', 'texture2': 'texture error', 'perimeter2': 'perimeter error', 'area2': 'area error',
        'smoothness2': 'smoothness error', 'compactness2': 'compactness error', 'concavity2': 'concavity error',
        'concave_points2': 'concave points error', 'symmetry2': 'symmetry error', 'fractal_dimension2': 'fractal dimension error',
        'radius3': 'worst radius', 'texture3': 'worst texture', 'perimeter3': 'worst perimeter', 'area3': 'worst area',
        'smoothness3': 'worst smoothness', 'compactness3': 'worst compactness', 'concavity3': 'worst concavity',
        'concave_points3': 'worst concave points', 'symmetry3': 'worst symmetry', 'fractal_dimension3': 'worst fractal dimension'
    }
    df = dataset.data.features.rename(columns=feature_mapping).copy()
    df['Diagnosis'] = dataset.data.targets['Diagnosis']
    print(f"Loaded via fetch_ucirepo (ID: 17) | Variable Types: {dataset.variables['type'].value_counts().to_dict()}")
except Exception:
    # Fallback to scikit-learn built-in loader if ucimlrepo is unavailable
    from sklearn.datasets import load_breast_cancer
    raw = load_breast_cancer(as_frame=True)
    df = raw.frame.copy()
    df['Diagnosis'] = np.where(df['target'] == 0, 'M', 'B')
    df = df.drop(columns=['target'])

print(f"Raw dataset shape: {df.shape}")
df.head()

------------------------------------------------------------------------

## Task 1: Data Preparation

- **Target Encoding & Missing Values:** Inspect the raw target
  `Diagnosis` and encode it such that $y = 1$ (Malignant) and $y = 0$
  (Benign). Check for and remove any missing values if present.
- **Feature Matrix & Class Imbalance:** Define the feature matrix
  $\mathbf{X}$ and target vector $y$. Investigate the class distribution
  and determine the malignant disease prevalence $\bar{p}$.
- **Stratified Split & Standardization:** Create a stratified train/test
  split (80% train, 20% test) and standardize the features using
  `StandardScaler`.

In [3]:
# Stratified train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Standardize features
scaler = StandardScaler()
# fit and transform training data
X_train_scaled = scaler.fit_transform(X_train)
# only transform test data
X_test_scaled = scaler.transform(X_test)

df_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols)
df_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set    : {X_test_scaled.shape[0]} samples")

------------------------------------------------------------------------

## Task 2: Baseline Model

- **Prevalence, Odds & Null Deviance:** Using the training data, define
  a baseline model with no predictors and only use the prevalence
  $\bar{p}$ for all observations. Calculate the odds, log-odds, and the
  null deviance of the training data
  $$D_0 = -2 \sum_{i=1}^n [y_i \ln \bar{p} + (1 - y_i)\ln(1 - \bar{p})]$$

- **Intercept-Only Logistic Regression:** Fit a logistic regression
  model with only an intercept term, extract the fitted parameter
  $\hat{\theta}_0$, and compare to the baseline log-odds.

- **Baseline Plot:** Plot the observed target values and the null model
  prediction $\hat{p} = \bar{p}$ for all samples (you may sort by sample
  index).

------------------------------------------------------------------------

## Task 3: Univariate Logistic Regression

- **Univariate Models & Ranking:** Conduct univariate logistic
  regressions for each feature $j$ independently. Compute the residual
  deviance $D(x_j)$ and the deviance drop relative to the null model,
  $\Delta D_j = D_0 - D(x_j)$. Display the ranking table.
- **Sigmoid & Linear Log-Odds:** For the top-ranked feature, plot both
  the predicted probability $\hat{p}(x)$ and the log-odds $z(x)$.
- **Parameter Interpretation & Threshold:** Interpret the slope
  parameter $\theta_1$ and determine the decision threshold $x^*$ where
  $\hat{p} = 0.5$.

------------------------------------------------------------------------

## Task 4: Bivariate Logistic Regression

- **Bivariate Model:** Fit a bivariate logistic regression using the top
  biomarker from Task 3 $x_1$ (`worst perimeter`) and a second biomarker
  $x_2$ (`worst texture`).
- **2D Probability Gradient & Decision Boundary:** Plot the predicted
  probability $\hat{p}$ across the 2D grid $(x_1, x_2)$ as a smooth
  contour gradient and plot the observed data points colored distinctly
  by disease status (Benign vs. Malignant).
- **Deviance Progression:** Calculate the residual deviance $D_2$ and
  confirm that $D_2 < D_1 < D_0$.

------------------------------------------------------------------------

## Task 5: Loss Landscape

- **Scan Parameter:** Fix the intercept $\theta_0 = \hat{\theta}_{0}$
  from Model 1 and scan the slope parameter $\theta_1$ across a broad
  range around the best prediction,
  $\theta_1 \in [-5 |\hat{\theta}_1|, +5 |\hat{\theta}_1|]$.
- **Loss Curves Comparison:** Compute and plot the Binary Cross-Entropy
  (BCE) and Mean Squared Error (MSE) for predictions
  $\hat{y}_i = \sigma(x_i^T \theta)$. What are the key differences in
  the asymptotic behavior of BCE vs. MSE?

------------------------------------------------------------------------

## Task 6: Multivariate Logistic Regression

- **Full Model Fitting:** Fit a multivariable logistic regression using
  all standardized features.
- **Deviance Comparison:** Compare the residual deviance $D$, and
  deviance drop $\Delta D$ for the four models (Model 0: Null, Model 1:
  Univariate, Model 2: Bivariate, Model 3: Full).

------------------------------------------------------------------------

## Task 7: Model Evaluation

- **Evaluation on the Test Set:** Evaluate the fitted full model
  (Model 3) on the unseen **test set** ($\mathbf{X}_{\text{test}}$,
  $\mathbf{y}_{\text{test}}$) using a decision threshold of $t = 0.5$.
  Calculate and visualize the Confusion Matrix using scikit-learn, and
  calculate Sensitivity (Recall), Specificity, Precision, and Accuracy.
- **Diagnostic Curves:** Plot the ROC and Precision-Recall curves for
  the test set predictions, including the AUC and Average Precision (AP)
  scores.

------------------------------------------------------------------------

## Task 8: Optional Regularization

The default implementation of `LogisticRegression()` in scikit-learn
applies $L_2$ **Ridge** regularization with a default penalty strength
of $C=1.0$. Explore the impact of this parameter on model performance.
An alternative regularization approach is $L_1$ **Lasso**, which can
drive redundant collinear features to exact zero, yielding a sparser
model. You can explore this with parameters `C`, `l1_ratio=1` and
`solver='saga'`. How many features remain active (non-zero) under Lasso?
Which are removed?

------------------------------------------------------------------------

## Checkpoint and Review Questions

- **Question 1 (Training vs. Test Data & Preprocessing Leakage):** Why
  must ML models be evaluated on an independent test set? Why is it
  critical that feature scaling (e.g. `StandardScaler`) is fitted
  strictly on the training partition?
- **Question 2 (Parameter Interpretation & Odds Ratios):** In logistic
  regression, how is a fitted slope parameter $\theta_j$ interpreted in
  terms of the Odds Ratio ($\text{OR}_j = \exp(\theta_j)$)? What does it
  mean for the odds if $\theta_j > 0$, $\theta_j = 0$, or
  $\theta_j < 0$?
- **Question 3 (Confusion Matrix & ROC):** In medical diagnostics, why
  is overall classification accuracy often misleading? Name two
  complementary metrics from the confusion matrix that distinguish
  between error types. Explain what the ROC curve shows. Why is AUC a
  useful summary statistic?